# EEI6366 Lab 01 (Alternative): Big Data Processing with PySpark

**Dataset:** `open_university_student_performance.csv`  
**Size:** 150,500 rows  

This notebook uses **simple and easy operations** for both PySpark and Spark SQL.

- Explore using **PySpark**: 4 easy operations
- Explore using **Spark SQL**: 4 easy operations

## 1. Install and Start PySpark

In [ ]:
!pip install -q pyspark findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Lab01-Alter").getOrCreate()
spark

## 2. Load the Dataset

In [ ]:
from google.colab import files
print("Upload open_university_student_performance.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = spark.read.csv(file_name, header=True, inferSchema=True)

print("Total rows:", df.count())
df.show(5)

## 3. PySpark Operations (4 Easy Operations)

### Operation 1 – Show Schema and Count

See the structure of the data and total number of rows.

In [ ]:
df.printSchema()
print("Total rows:", df.count())

### Operation 2 – Filter Data

Show only the students who **Failed**.

In [ ]:
failed_students = df.filter(df.status == "Fail")
failed_students.show(10)
print("Number of Fail students:", failed_students.count())

### Operation 3 – Select Columns and Order

Select only a few columns and sort by overall mark (highest first).

In [ ]:
df.select("student_id", "course_code", "overall_mark", "grade") \
  .orderBy(F.desc("overall_mark")) \
  .show(10)

### Operation 4 – GroupBy and Average

Find the average overall mark for each center.

In [ ]:
df.groupBy("center") \
  .avg("overall_mark") \
  .show()

## 4. Spark SQL Operations (4 Easy Operations)

First create a temporary view so we can use SQL.

In [ ]:
df.createOrReplaceTempView("students")

### SQL Operation 1 – Simple SELECT with WHERE

Show students from Colombo center who got grade A or A+.

In [ ]:
spark.sql("""
SELECT student_id, course_code, overall_mark, grade
FROM students
WHERE center = 'Colombo' AND grade IN ('A', 'A+')
LIMIT 10
""").show()

### SQL Operation 2 – COUNT by Status

Count how many students Passed and how many Failed.

In [ ]:
spark.sql("""
SELECT status, COUNT(*) AS total
FROM students
GROUP BY status
""").show()

### SQL Operation 3 – Average Mark by Course

Find the average overall mark for each course.

In [ ]:
spark.sql("""
SELECT course_code, ROUND(AVG(overall_mark), 2) AS avg_mark
FROM students
GROUP BY course_code
ORDER BY avg_mark DESC
""").show()

### SQL Operation 4 – Highest Mark in Each Center

Find the maximum overall mark for every center.

In [ ]:
spark.sql("""
SELECT center, MAX(overall_mark) AS highest_mark
FROM students
GROUP BY center
ORDER BY highest_mark DESC
""").show()

## 5. Short Summary (for Viva)

**PySpark Operations used:**
1. `printSchema()` and `count()` → understand data structure
2. `filter()` → select only Fail students
3. `select()` + `orderBy()` → choose columns and sort
4. `groupBy()` + `avg()` → average mark by center

**Spark SQL Operations used:**
1. `SELECT ... WHERE` → filter Colombo A/A+ students
2. `GROUP BY status` + `COUNT` → Pass vs Fail count
3. `AVG()` by course → average mark per course
4. `MAX()` by center → highest mark in each center

All operations are simple and easy to explain.

In [ ]:
spark.stop()
print("Spark stopped.")